# 04 — Bonus Çözümü: Görev Atama

> **Bu dosya `solutions` branch'indedir.**

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q pulp
    !git clone -q -b solutions https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import pandas as pd
import matplotlib.pyplot as plt
from utils.plotting import plot_gantt

proc_times = {
    0: {0: 3, 1: 5, 2: 2},
    1: {0: 7, 1: 4, 2: 6},
    2: {0: 2, 1: 3, 2: 8},
    3: {0: 5, 1: 6, 2: 3},
    4: {0: 4, 1: 2, 2: 5},
    5: {0: 6, 1: 7, 2: 4},
    6: {0: 3, 1: 4, 2: 6},
    7: {0: 8, 1: 3, 2: 2},
}
jobs     = list(proc_times.keys())
machines = list(proc_times[0].keys())

## Temel Model — Makespan Minimizasyonu

In [ ]:
prob = pulp.LpProblem("task_assignment", pulp.LpMinimize)

x = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
     for j in jobs for m in machines}
C_max = pulp.LpVariable("C_max", lowBound=0, cat="Continuous")

prob += C_max
for j in jobs:
    prob += pulp.lpSum(x[j, m] for m in machines) == 1
for m in machines:
    prob += pulp.lpSum(proc_times[j][m] * x[j, m] for j in jobs) <= C_max

prob.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Durum   : {pulp.LpStatus[prob.status]}")
print(f"Makespan: {pulp.value(C_max):.1f}")

assignment = {j: m for j in jobs for m in machines if pulp.value(x[j, m]) > 0.5}
for j in jobs:
    m = assignment[j]
    print(f"  Görev {j} → İşlemci {m}  (süre: {proc_times[j][m]})")

fig = plot_gantt(assignment, proc_times, num_machines=len(machines))
plt.show()

## Görev 2 — Yasak Atama: Görev 0 İşlemci 2'de Çalışamaz

In [ ]:
prob_g2 = pulp.LpProblem("forbidden", pulp.LpMinimize)
x2 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
      for j in jobs for m in machines}
cmax2 = pulp.LpVariable("C_max", lowBound=0)

prob_g2 += cmax2
for j in jobs:
    prob_g2 += pulp.lpSum(x2[j, m] for m in machines) == 1
for m in machines:
    prob_g2 += pulp.lpSum(proc_times[j][m] * x2[j, m] for j in jobs) <= cmax2
prob_g2 += x2[0, 2] == 0  # YASAK

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
print(f"Yasak atama ile makespan: {pulp.value(cmax2):.1f}")
print(f"Orijinal makespan       : {pulp.value(C_max):.1f}")

## Görev 3 — Objective Karşılaştırması: Makespan vs Toplam Süre

In [ ]:
prob_g3 = pulp.LpProblem("total_time", pulp.LpMinimize)
x3 = {(j, m): pulp.LpVariable(f"x_{j}_{m}", cat="Binary")
      for j in jobs for m in machines}

prob_g3 += pulp.lpSum(
    proc_times[j][m] * x3[j, m] for j in jobs for m in machines
)
for j in jobs:
    prob_g3 += pulp.lpSum(x3[j, m] for m in machines) == 1

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))

total_v = int(pulp.value(prob_g3.objective))
makespan_g3 = max(
    sum(proc_times[j][m] for j in jobs if pulp.value(x3[j, m]) > 0.5)
    for m in machines
)

print(f"Toplam süre minimize → toplam süre : {total_v}")
print(f"Bu çözümün makespan'i              : {makespan_g3}")
print(f"Orijinal makespan min. makespan    : {int(pulp.value(C_max))}")

fig_g3 = plot_gantt(
    {j: m for j in jobs for m in machines if pulp.value(x3[j, m]) > 0.5},
    proc_times, num_machines=len(machines)
)
plt.title("Toplam Süre Minimize — Gantt")
plt.show()